# 03 — Run Gemma 4 26B-A4B APEX I-Mini

**Gemma 4 26B-A4B** is a 26-billion-parameter MoE model (activates ~4B per token). The **APEX I-Mini** quantization brings it to 12.27 GB — fitting on a free T4.

| Metric | Value |
|--------|-------|
| File size | 12.27 GB |
| Load time | 98.4s |
| Status | ✅ Proven to fit on T4 |

## 1. Install llama-cpp-python with CUDA

In [ ]:
%%capture
pip install llama-cpp-python huggingface_hub --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124


## 2. Download the model

In [ ]:
from huggingface_hub import hf_hub_download, list_repo_files
import os, time

# Update repo_id and filename to the actual APEX I-Mini GGUF repo
repo_id = "mudler/Carnice-Gemma4-26B-A4B-APEX-GGUF"  # adjust as needed

# List files to find the exact shard name
files = list_repo_files(repo_id)
gguf_files = [f for f in files if f.endswith('.gguf')]
print("Available GGUF files:")
for f in gguf_files:
    print(f"  {f}")

# Use the first shard (llama.cpp auto-loads the rest)
filename = gguf_files[0] if gguf_files else None
assert filename, "No GGUF files found in repo"

t0 = time.time()
model_path = hf_hub_download(repo_id=repo_id, filename=filename)
print(f"\nDownloaded in {time.time()-t0:.1f}s")
print(f"Path: {model_path}")


## 3. Load onto the GPU

In [ ]:
from llama_cpp import Llama
import time

t0 = time.time()
llm = Llama(
    model_path=model_path,
    n_gpu_layers=-1,
    n_ctx=8192,
    verbose=False,
)
print(f"\nModel loaded in {time.time()-t0:.1f}s")


## 4. Check VRAM

In [ ]:
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv


## 5. Run inference

In [ ]:
prompt = "Give me 5 creative names for a coffee shop that also sells books, with a one-line rationale for each."

resp = llm(
    prompt,
    max_tokens=400,
    temperature=0.8,
    verbose=False,
)
print(resp['choices'][0]['text'])


## About APEX I-Mini

APEX I-Mini is a slightly higher-precision variant of APEX Nano, tuned for models like Gemma 4 26B-A4B where the MoE structure benefits from keeping expert routing weights a bit wider. Still ultra-low-bit, still fits in 15 GB.